In [1]:
print("Hello, World!")

Hello, World!


In [11]:
!pip install pandas numpy scikit-learn imbalanced-learn nltk lxml torch

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 10.4 MB/s eta 0:00:0000:0100:01
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.4 MB/s eta 0:00:00
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached markupsafe-3.0.3-cp312-cp312-macosx_11_0_arm64.whl (12 kB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To upda

In [12]:
import xml.etree.ElementTree as ET
import pandas as pd
import torch

In [13]:
# Load predator list (training)
predator_train_path = "/Users/tiarasabrina/Downloads/pan12-sexual-predator-identification-test-and-training/pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-predators-2012-05-01.txt"

with open(predator_train_path, 'r') as f:
    predators_train = set(line.strip() for line in f if line.strip())

print(f"Total predator authors (train): {len(predators_train)}")

# Parse XML train
xml_train_path = "/Users/tiarasabrina/Downloads/pan12-sexual-predator-identification-test-and-training/pan12-sexual-predator-identification-training-corpus-2012-05-01/pan12-sexual-predator-identification-training-corpus-2012-05-01.xml"

def parse_xml(xml_path, predator_set):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    records = []
    for conv in root.findall('conversation'):
        conv_id = conv.get('id')
        for msg in conv.findall('message'):
            author_el = msg.find('author')
            text_el   = msg.find('text')

            author = author_el.text.strip() if author_el is not None and author_el.text else 'unknown'
            text   = text_el.text.strip()   if text_el   is not None and text_el.text   else ''

            records.append({
                'conv_id'     : conv_id,
                'author'      : author,
                'text'        : text,
                'is_predator' : 1 if author in predator_set else 0
            })

    return pd.DataFrame(records)

df_train = parse_xml(xml_train_path, predators_train)

print(f"\nShape df_train: {df_train.shape}")
print("\n=== TRAIN ===")
print(df_train['is_predator'].value_counts())
print(f"  Non-predator : {(df_train['is_predator'] == 0).sum()}")
print(f"  Predator     : {(df_train['is_predator'] == 1).sum()}")
df_train.head()

Total predator authors (train): 142

Shape df_train: (903607, 4)

=== TRAIN ===
is_predator
0    862629
1     40978
Name: count, dtype: int64
  Non-predator : 862629
  Predator     : 40978


,conv_id,author,text,is_predator
0,e621da5de598c9321a1d505ea95e6a2d,97964e7a9e8eb9cf78f2e4d7b2ff34c7,Hola.,0
1,e621da5de598c9321a1d505ea95e6a2d,0158d0d6781fc4d493f243d4caa49747,hi.,0
2,e621da5de598c9321a1d505ea95e6a2d,0158d0d6781fc4d493f243d4caa49747,whats up?,0
3,e621da5de598c9321a1d505ea95e6a2d,97964e7a9e8eb9cf78f2e4d7b2ff34c7,not a ton.,0
4,e621da5de598c9321a1d505ea95e6a2d,97964e7a9e8eb9cf78f2e4d7b2ff34c7,you?,0


In [ ]:
# Load groundtruth problem 1
gt_path1 = "/Users/tiarasabrina/Downloads/pan12-sexual-predator-identification-test-and-training/pan12-sexual-predator-identification-test-corpus-2012-05-21/pan12-sexual-predator-identification-groundtruth-problem1.txt"

with open(gt_path1, 'r') as f:
    predators_test = set(line.strip() for line in f if line.strip())

print(f"Total predator authors (test): {len(predators_test)}")

# Parse XML test
xml_test_path = "/Users/tiarasabrina/Downloads/pan12-sexual-predator-identification-test-and-training/pan12-sexual-predator-identification-test-corpus-2012-05-21/pan12-sexual-predator-identification-test-corpus-2012-05-17.xml"

df_test = parse_xml(xml_test_path, predators_test)

print(f"\nShape df_test: {df_test.shape}")
print("\n=== TEST ===")
print(df_test['is_predator'].value_counts())
print(f"  Non-predator : {(df_test['is_predator'] == 0).sum()}")
print(f"  Predator     : {(df_test['is_predator'] == 1).sum()}")
df_test.head()

Total predator authors (test): 254

Shape df_test: (2058781, 4)

=== TEST ===
is_predator
0    1993542
1      65239
Name: count, dtype: int64
  Non-predator : 1993542
  Predator     : 65239


,conv_id,author,text,is_predator
0,affc2df0951b733d14ba92d19d9b7695,0a39f78bcb297ab0ebe8a29c28bfed89,bugmail: [Bug 6978] New: Mark eof-terminated s...,0
1,affc2df0951b733d14ba92d19d9b7695,60659cfda992013e610f285c46692d28,"Henri, can I ask you a Firefox build question ...",0
2,affc2df0951b733d14ba92d19d9b7695,b8810fee2f4a71f849f3f7409546d1d9,"60659cfda992013e610f285c46692d28: sure, but I ...",0
3,affc2df0951b733d14ba92d19d9b7695,60659cfda992013e610f285c46692d28,"It appears the build runs through, it creates ...",0
4,affc2df0951b733d14ba92d19d9b7695,60659cfda992013e610f285c46692d28,"when I start it, I get my standard install of ...",0


In [17]:
torch.save(df_train, "df_train.pth")
torch.save(df_test,  "df_test.pth")

print("Saved: df_train.pth & df_test.pth")

Saved: df_train.pth & df_test.pth


In [19]:
df_train_loaded = torch.load("df_train.pth", weights_only=False)
df_test_loaded  = torch.load("df_test.pth", weights_only=False)

print("=== TRAIN ===")
print(f"  Shape        : {df_train_loaded.shape}")
print(f"  Non-predator : {(df_train_loaded['is_predator'] == 0).sum()}")
print(f"  Predator     : {(df_train_loaded['is_predator'] == 1).sum()}")

print("\n=== TEST ===")
print(f"  Shape        : {df_test_loaded.shape}")
print(f"  Non-predator : {(df_test_loaded['is_predator'] == 0).sum()}")
print(f"  Predator     : {(df_test_loaded['is_predator'] == 1).sum()}")

=== TRAIN ===
  Shape        : (903607, 4)
  Non-predator : 862629
  Predator     : 40978

=== TEST ===
  Shape        : (2058781, 4)
  Non-predator : 1993542
  Predator     : 65239
